# augmented-v5: günlük tanıtım ve istek diliyle eğitim turu

**Çalışma zamanı → T4 GPU → Tümünü çalıştır.** Üç seed, her biri yaklaşık 1 saat. Her seed bitince Drive'a kaydedilir;
bağlantı koparsa ya da GPU kotası biterse yeniden **Tümünü çalıştır**: hazır olanlar atlanır.
Veri: Drive'daki `turkce-dlp-colab-bundle-v7` paketi (augmented-v5). Değerlendirme yalnız geliştirme setinde
(realistic-v3, üç yazım biçimi); mühürlü setler burada çalışmaz.


In [ ]:
import os, pathlib, shutil, subprocess, zipfile, json, torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı türünü T4 GPU yap."
from google.colab import drive
drive.mount('/content/drive')
# Klasörler başka bir hesaptan paylaşılıp kısayol olarak eklenmiş olabilir: kısayol hedeflerinde de ara
ROOTS = [pathlib.Path('/content/drive/MyDrive'), pathlib.Path('/content/drive/.shortcut-targets-by-id')]
def walk_dirs():
    for r in ROOTS:
        if r.exists():
            for d, subdirs, files in os.walk(r, followlinks=True):
                yield pathlib.Path(d), subdirs, files
DRIVE = pathlib.Path('/content/drive/MyDrive/turkce-llm-dlp')
if not DRIVE.exists():
    DRIVE = next((d / 'turkce-llm-dlp' for d, sub, _ in walk_dirs() if 'turkce-llm-dlp' in sub), DRIVE)
assert (DRIVE / 'runs').exists(), f'{DRIVE}/runs bulunamadı: turkce-llm-dlp klasörünü bu hesapta Drive\'ım\'a kısayol olarak ekle'
print('Drive klasörü:', DRIVE)
os.chdir('/content')
shutil.rmtree('/content/proje', ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/irmakyaprakk/turkce-llm-dlp.git', '/content/proje'], check=True)
bundles = set()
for d, _, files in walk_dirs():
    if 'BUNDLE_INFO.txt' in files: bundles.add(d)
    bundles |= {d / f for f in files if f.startswith('turkce-dlp-colab-bundle') and f.endswith('.zip')}
for src in sorted(bundles, key=lambda p: p.name):
    tmp = pathlib.Path('/content/_paket'); shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src, tmp) if src.is_dir() else zipfile.ZipFile(src).extractall(tmp)
    root = next((p.parent for p in tmp.rglob('BUNDLE_INFO.txt')), tmp)
    if (root / 'data').exists():
        shutil.copytree(root / 'data', '/content/proje/data', dirs_exist_ok=True)
os.chdir('/content/proje')
!pip -q install -r training/ner/requirements.txt
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# augmented-v5 verisi: Drive'daki v7 paketinden gelir (yoksa kaynaklardan yeniden üretilir); dondurulan sürümle doğrulanır
import hashlib
if not pathlib.Path('data/ner/processed/augmented-v5/train.jsonl').exists():
    for need in ('data/ner/processed/person-org-location-v1/train.jsonl', 'data/ner/processed/openstreetmap-tr/splits/train.jsonl',
                 'data/ner/processed/real-address-v2/test.jsonl'):
        assert pathlib.Path(need).exists(), f"augmented-v5 yok: 'turkce-dlp-colab-bundle-v7' paketini Drive'a yükle ({need} de bulunamadı)"
    !python training/ner/prepare_augmented.py --osm-original-ratio 0.15 --v3-contexts 10000 800 --mention-replace 5000 300 --v4-informal 12000 800 --v5-informal 8000 600 --output-dir data/ner/processed/augmented-v5 > /dev/null
EXPECTED = {'train': '7e550fe058db64784a76c43017bf8d5403d2d12c6e1b613a13333c682a2cea58', 'dev': '031f3c7e18d2d8529de9a28993b28eaa54b965997b268b2037466c067ebe8862'}
for split, digest in EXPECTED.items():
    got = hashlib.sha256(pathlib.Path(f'data/ner/processed/augmented-v5/{split}.jsonl').read_bytes()).hexdigest()
    assert got == digest, f'{split} verisi dondurulan sürümle aynı değil ({got[:12]})'
print('augmented-v5 doğrulandı:', json.load(open('data/ner/processed/augmented-v5/quality_report.json'))['rows'])


In [ ]:
# Üç seed eğitimi (her biri ~1 saat). Hazır olan atlanır; kesilirse yeniden "Tümünü çalıştır".
for seed in (42, 43, 44):
    saved = DRIVE / 'runs' / f'augmented_v5_seed{seed}'
    if (saved / 'model').exists():
        print(f'seed {seed}: hazır, atlanıyor'); continue
    out = f'/content/runs/augmented_v5_seed{seed}'
    print(f'seed {seed}: eğitim başlıyor')
    !USE_TF=0 python training/ner/train.py --data-dir data/ner/processed/augmented-v5 --labels-path training/ner/labels.json --output-dir {out} --epochs 3 --seed {seed} --fp16 > /content/train-{seed}.log 2>&1
    !tail -3 /content/train-{seed}.log
    shutil.rmtree(saved, ignore_errors=True)
    shutil.copytree(out, saved, ignore=shutil.ignore_patterns('checkpoints'))
    print(f'seed {seed}: Drive\'a kaydedildi')


In [ ]:
# Geliştirme seti (realistic-v3 = batch1-3) üç yazım biçiminde: tümü ve yalnız batch3 (günlük dil, şablonlarla aynı kaynak → iyimser)
import statistics
!python training/ner/build_gold_set.py --out-dir data/ner/processed/realistic-v3 > /dev/null
results = {}
for seed in (42, 43, 44):
    model = DRIVE / 'runs' / f'augmented_v5_seed{seed}' / 'model'
    for name, extra in (('tum', ''), ('batch3', '--batch realistic_batch3')):
        target = DRIVE / 'runs' / f'augmented_v5_seed{seed}' / f'casing-{name}.json'
        !USE_TF=0 python training/ner/evaluate_casing.py --model-dir "{model}" --data-dir data/ner/processed/realistic-v3 {extra} --device 0 --output-json "{target}" > /dev/null
        results.setdefault(name, []).append(json.loads(target.read_text()))
summary = {}
for name, runs in results.items():
    for variant in ('original', 'lower', 'upper'):
        values = [r['variants'][variant]['hybrid']['metrics']['overall_f1'] for r in runs]
        summary[f'{name}/{variant}'] = f"{statistics.mean(values):.3f} ± {statistics.stdev(values):.3f}"
(DRIVE / 'runs' / 'casing-v5-summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=1))
print('hibrit F1 (ortalama ± std, 3 seed):'); print(json.dumps(summary, ensure_ascii=False, indent=1))
